In [2]:
from pathlib import Path
from gears import PertData
import pandas as pd
import scanpy as sc
from tqdm import tqdm

In [3]:
ref_dir = Path('/Users/djemec/data/jepa/reference_data')
data_dir = Path('/Users/djemec/data/jepa/v0_6')
pert_dir = data_dir / 'pert_embd'
training_dir = data_dir / 'training'
pretraining_dir = data_dir / 'pretraining'

pert_temp = data_dir / 'pert_tmp'

In [4]:
key_ds =  {'k562e':ref_dir / 'k562e'}
datasets= {
    'k562e_raw': ref_dir / 'raw_k562e' / 'K562_essential_raw_singlecell_01.h5ad',
    'rep1e':ref_dir /'rep1e'/'rpe1_raw_singlecell_01.h5ad',
    'k562gw':ref_dir /'k562gw'/'perturb_processed.h5ad',
    'adamson':ref_dir /'adamson'/'AdamsonWeissman2016_GSM2406681_10X010.h5ad',
    'norman':ref_dir /'norman'/'NormanWeissman2019_filtered.h5ad',
    'sciplex':ref_dir /'sciplex'/'SrivatsanTrapnell2020_sciplex3.h5ad', 
}

In [11]:
splits= ['train','val','test']
n_genes = 16384
count_normalize_target = 1e4

## Review Key dataset and extract genes we have counts for

In [32]:
ds_key = 'k562e_raw'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df.head()

,gene_name,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano
gene_id,,,,,,,,,,,,
ENSG00000237491,LINC01409,chr1,778747,810065,gene_version10,+,31318,True,0.137594,0.380048,2.762105,1.049733
ENSG00000228794,LINC01128,chr1,825138,868202,gene_version9,+,43064,True,0.256720,0.520162,2.026184,1.053944
ENSG00000188976,NOC2L,chr1,944203,959309,gene_version11,-,15106,True,1.975144,1.707837,0.864665,1.476706
ENSG00000187961,KLHL17,chr1,960584,965719,gene_version14,+,5135,True,0.119593,0.353702,2.957540,1.046089
ENSG00000188290,HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.249577,0.561933,2.251540,1.265214


In [33]:
all_genes = var_df['gene_name'].to_dict()
len(all_genes.keys())

8563

## Get Genes from other datasets

#### rep1e

In [34]:
ds_key = 'rep1e'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df.head()

,gene_name,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano
gene_id,,,,,,,,,,,,
ENSG00000188976,NOC2L,chr1,944203,959309,gene_version11,-,15106,True,0.997140,1.149519,1.152816,1.325185
ENSG00000187583,PLEKHN1,chr1,966482,975865,gene_version11,+,9383,True,0.131328,0.377933,2.877783,1.087609
ENSG00000188290,HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.732129,1.526109,2.084482,3.181148
ENSG00000187608,ISG15,chr1,1001138,1014540,gene_version10,+,13402,True,0.455956,1.260789,2.765152,3.486272
ENSG00000188157,AGRN,chr1,1020120,1056118,gene_version15,+,35998,True,0.346108,0.648378,1.873341,1.214633


In [35]:
new_genes = var_df['gene_name'].to_dict()
len(new_genes.keys())

8749

In [36]:
# check conflicts
conflicts = {k: (all_genes[k], v) for k, v in new_genes.items() if k in all_genes and all_genes[k] != v}
conflicts

{}

In [37]:
all_genes = {**new_genes, **all_genes}
len(all_genes)

10086

#### k562gw

In [43]:
ds_key = 'k562gw'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df = var_df[var_df.ncells >0]
var_df = var_df[var_df.ncounts >0]
var_df.head()

,chr,start,end,class,strand,length,in_matrix,mean,std,cv,fano,ensembl_id,ncounts,ncells
gene_name,,,,,,,,,,,,,,
LINC01409,chr1,778747,810065,gene_version10,+,31318,True,0.116626,0.349971,3.000803,1.050194,ENSG00000237491,232036.0,214242
LINC01128,chr1,825138,868202,gene_version9,+,43064,True,0.182850,0.437274,2.391434,1.045713,ENSG00000228794,363795.0,325879
NOC2L,chr1,944203,959309,gene_version11,-,15106,True,1.415674,1.397208,0.986957,1.378984,ENSG00000188976,2816593.0,1391744
KLHL17,chr1,960584,965719,gene_version14,+,5135,True,0.105599,0.330678,3.131439,1.035497,ENSG00000187961,210098.0,196093
HES4,chr1,998962,1000172,gene_version10,-,1210,True,0.242700,0.550596,2.268630,1.249098,ENSG00000188290,482870.0,391564


In [44]:
new_genes = var_df.reset_index().set_index('ensembl_id')['gene_name'].to_dict()
len(new_genes.keys())

8248

In [45]:
# check conflicts, these are fine to overwrite
conflicts = {k: (all_genes[k], v) for k, v in new_genes.items() if k in all_genes and all_genes[k] != v}
conflicts

{'ENSG00000285053': ('TBCE', 'TBCE_ENSG00000285053'),
 'ENSG00000284770': ('TBCE', 'TBCE_ENSG00000284770'),
 'ENSG00000284024': ('HSPA14', 'HSPA14_ENSG00000284024'),
 'ENSG00000187522': ('HSPA14', 'HSPA14_ENSG00000187522')}

In [46]:
all_genes = {**new_genes, **all_genes}
len(all_genes)

10117

#### adamson

Review as this will be the smallest

In [47]:
ds_key = 'adamson'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df = var_df[var_df.ncells >0]
var_df = var_df[var_df.ncounts >0]
var_df.head()

,ensembl_id,ncounts,ncells
gene_symbol,,,
MIR1302-10,ENSG00000243485,11.0,11
RP11-34P13.8,ENSG00000239945,43.0,43
AL627309.1,ENSG00000237683,337.0,333
AP006222.2,ENSG00000228463,48.0,48
RP4-669L17.10,ENSG00000237094,2.0,2


In [48]:
new_genes = var_df.reset_index().set_index('ensembl_id')['gene_symbol'].to_dict()
len(new_genes.keys())

23376

In [49]:
# check conflicts, these are fine to overwrite
conflicts = {k: (all_genes[k], v) for k, v in new_genes.items() if k in all_genes and all_genes[k] != v}
conflicts

{'ENSG00000237491': ('LINC01409', 'RP11-206L10.9'),
 'ENSG00000127054': ('INTS11', 'CPSF3L'),
 'ENSG00000224051': ('CPTP', 'GLTPD1'),
 'ENSG00000224870': ('MRPL20-AS1', 'RP4-758J18.2'),
 'ENSG00000228594': ('FNDC10', 'C1orf233'),
 'ENSG00000162585': ('FAAP20', 'C1orf86'),
 'ENSG00000157870': ('PRXL2B', 'FAM213B'),
 'ENSG00000231789': ('PIK3CD-AS2', 'RP11-558F24.4'),
 'ENSG00000175279': ('CENPS', 'APITD1'),
 'ENSG00000179743': ('AL450998.2', 'RP11-169K16.9'),
 'ENSG00000040487': ('SLC66A1', 'PQLC2'),
 'ENSG00000173436': ('MICOS10', 'MINOS1'),
 'ENSG00000261326': ('LINC01355', 'RP5-1057J7.6'),
 'ENSG00000011007': ('ELOA', 'TCEB3'),
 'ENSG00000236810': ('ELOA-AS1', 'RP5-886K2.3'),
 'ENSG00000117616': ('RSRP1', 'C1orf63'),
 'ENSG00000204178': ('MACO1', 'TMEM57'),
 'ENSG00000162430': ('SELENON', 'SEPN1'),
 'ENSG00000130770': ('ATP5IF1', 'ATPIF1'),
 'ENSG00000243749': ('TMEM35B', 'ZMYM6NB'),
 'ENSG00000183520': ('UTP11', 'UTP11L'),
 'ENSG00000259943': ('AL050341.2', 'RP1-39G22.7'),
 'ENSG000

In [50]:
all_genes = {**new_genes, **all_genes}
len(all_genes)

23750

#### norman

In [51]:
ds_key = 'norman'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df = var_df[var_df.ncells >0]
var_df = var_df[var_df.ncounts >0]
var_df.head()

,ensemble_id,ncounts,ncells
RP11-34P13.3,ENSG00000243485,29.0,29
RP11-34P13.7,ENSG00000238009,266.0,265
RP11-34P13.8,ENSG00000239945,10.0,10
RP11-34P13.9,ENSG00000241599,1.0,1
FO538757.3,ENSG00000279928,12.0,12


In [53]:
new_genes = var_df.reset_index().set_index('ensemble_id')['index'].to_dict()
len(new_genes.keys())

25038

In [55]:
# check conflicts, these are fine to overwrite
conflicts = {k: (all_genes[k], v) for k, v in new_genes.items() if k in all_genes and all_genes[k] != v}
conflicts

{'ENSG00000243485': ('MIR1302-10', 'RP11-34P13.3'),
 'ENSG00000237491': ('LINC01409', 'RP11-206L10.9'),
 'ENSG00000187642': ('C1orf170', 'PERM1'),
 'ENSG00000223823': ('RP11-465B22.5', 'LINC01342'),
 'ENSG00000127054': ('INTS11', 'CPSF3L'),
 'ENSG00000215014': ('AL645728.1', 'RP5-832C2.5'),
 'ENSG00000228594': ('FNDC10', 'C1orf233'),
 'ENSG00000142609': ('C1orf222', 'CFAP74'),
 'ENSG00000157870': ('PRXL2B', 'FAM213B'),
 'ENSG00000233304': ('RP13-614K11.1', 'LINC01346'),
 'ENSG00000179840': ('C1orf200', 'PIK3CD-AS1'),
 'ENSG00000175279': ('CENPS', 'APITD1'),
 'ENSG00000179743': ('AL450998.2', 'FLJ37453'),
 'ENSG00000233421': ('U1', 'RP5-875O13.1'),
 'ENSG00000228549': ('U1-1', 'RP11-108M9.3'),
 'ENSG00000040487': ('SLC66A1', 'PQLC2'),
 'ENSG00000173436': ('MICOS10', 'MINOS1'),
 'ENSG00000249087': ('C1orf213', 'ZNF436-AS1'),
 'ENSG00000011007': ('ELOA', 'TCEB3'),
 'ENSG00000236810': ('ELOA-AS1', 'TCEB3-AS1'),
 'ENSG00000204178': ('MACO1', 'TMEM57'),
 'ENSG00000162430': ('SELENON', 'SEPN1

In [56]:
all_genes = {**new_genes, **all_genes}
len(all_genes)

27001

#### sciplex

In [59]:
ds_key = 'sciplex'
ds_adata = sc.read_h5ad(datasets[ds_key], backed='r')
var_df = ds_adata.var
var_df.head()

,ensembl_id
gene_symbol,
TSPAN6,ENSG00000000003
TNMD,ENSG00000000005
DPM1,ENSG00000000419
SCYL3,ENSG00000000457
C1orf112,ENSG00000000460


In [60]:
new_genes = var_df.reset_index().set_index('ensembl_id')['gene_symbol'].to_dict()
len(new_genes.keys())

110938

In [61]:
# check conflicts, these are fine to overwrite
conflicts = {k: (all_genes[k], v) for k, v in new_genes.items() if k in all_genes and all_genes[k] != v}
conflicts

{'ENSG00000002586': ('CD99', 'CD99:1'),
 'ENSG00000005379': ('BZRAP1', 'TSPOAP1'),
 'ENSG00000006015': ('REX1BD', 'C19orf60'),
 'ENSG00000010165': ('EEF1AKNMT', 'METTL13'),
 'ENSG00000011052': ('NME2', 'NME1-NME2'),
 'ENSG00000020181': ('GPR124', 'ADGRA2'),
 'ENSG00000022277': ('RTF2', 'RTFDC1'),
 'ENSG00000036549': ('AC118549.1', 'ZZZ3'),
 'ENSG00000039123': ('MTREX', 'SKIV2L2'),
 'ENSG00000040487': ('SLC66A1', 'PQLC2'),
 'ENSG00000050030': ('KIAA2022', 'NEXMIF'),
 'ENSG00000063169': ('GLTSCR1', 'BICRA'),
 'ENSG00000064489': ('MEF2BNB-MEF2B', 'BORCS8-MEF2B'),
 'ENSG00000065600': ('PACC1', 'TMEM206'),
 'ENSG00000069122': ('GPR116', 'ADGRF5'),
 'ENSG00000081791': ('DELE1', 'KIAA0141'),
 'ENSG00000082074': ('FYB', 'FYB1'),
 'ENSG00000083097': ('DOP1A', 'DOPEY1'),
 'ENSG00000083223': ('TUT7', 'ZCCHC6'),
 'ENSG00000084444': ('KIAA1467', 'FAM234B'),
 'ENSG00000086619': ('ERO1LB', 'ERO1B'),
 'ENSG00000087302': ('RTRAF', 'C14orf166'),
 'ENSG00000089101': ('C20orf26', 'CFAP61'),
 'ENSG00000093

In [62]:
all_genes = {**new_genes, **all_genes}
len(all_genes)

111820